In [0]:
print("Spark version:", spark.version)

In [0]:
display(spark.sql("SHOW CATALOGS"))


In [0]:
CATALOG = "smart-trading-dev"
display(spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`"))

In [0]:
VOLUME = "00_volume"
BRONZE = "01_bronze"
SILVER = "02_silver"
GOLD = "03_gold"

display(
    spark.sql(f"SHOW TABLES IN `{CATALOG}`.{BRONZE}")
)

In [0]:
RAW_AGGTRADES = "raw_aggtrades"
RAW_NEWS = "raw_news"

In [0]:
RAW_AGGTRADES_PATH = f"`{CATALOG}`.`{BRONZE}`.`{RAW_AGGTRADES}`"
RAW_NEWS_PATH = f"`{CATALOG}`.`{BRONZE}`.`{RAW_NEWS}`"
print(f"RAW_AGGTRADES_PATH: {RAW_AGGTRADES_PATH}")
print(f"RAW_NEWS_PATH: {RAW_NEWS_PATH}")

In [0]:
raw_news_df = spark.table(RAW_NEWS_PATH)
display(raw_news_df.limit(2,))

### 1. Load and sample from RAW_NEWS_PATH

In [0]:
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient

preferred = [
    # "databricks-meta-llama-3-3-70b-instruct",
    # "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
    
]

w = WorkspaceClient()
available = {ep.name for ep in w.serving_endpoints.list()}

fallback_candidates = sorted(name for name in available if name.startswith("databricks-"))
MODEL_ENDPOINT = next(
    (name for name in preferred if name in available),
    fallback_candidates[0] if fallback_candidates else None,
 )

if MODEL_ENDPOINT is None:
    raise RuntimeError(
        "No Databricks FM endpoint is available in this workspace. "
        f"Checked preferred={preferred}. "
        f"Available endpoints={sorted(available)}"
    )

print("Using MODEL_ENDPOINT:", MODEL_ENDPOINT)

In [0]:
SAMPLE_N = 20

raw_news_df = (
    spark.table(RAW_NEWS_PATH)
    .select(
        "story_id",
        "headline",
        "body",
        "source",
        "source_feed",
        "provider",
        "published_at",
        "url",
        "batch_id",
    )
    .where("headline IS NOT NULL")
    .where("body IS NOT NULL")
    .where("trim(headline) <> ''")
    .where("trim(body) <> ''")
    .limit(SAMPLE_N)
)

raw_news_df.createOrReplaceTempView("raw_news_ai_demo")
display(raw_news_df)

### 2. Build canonical text field for AI functions

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW raw_news_ai_text AS
SELECT
  story_id,
  headline,
  body,
  source,
  source_feed,
  provider,
  to_timestamp(published_at) AS published_at_ts,
  url,
  batch_id,
  concat_ws(' ', coalesce(headline, ''), coalesce(body, '')) AS news_text
FROM raw_news_ai_demo
""")

display(spark.sql("""
SELECT story_id, headline, published_at_ts, source_feed
FROM raw_news_ai_text
LIMIT 10
"""))

### 3. Sentiment with ai_analyze_sentiment

In [0]:
display(spark.sql("""
SELECT
  story_id,
  source_feed,
  ai_analyze_sentiment(news_text) AS sentiment,
  headline
FROM raw_news_ai_text
LIMIT 50
"""))

### 4. Market direction classification with ai_classify

In [0]:
display(spark.sql("""
SELECT
  story_id,
  ai_classify(news_text, array('bullish', 'bearish', 'neutral')) AS market_bias,
  headline
FROM raw_news_ai_text
LIMIT 50
"""))

### 5. Entity extraction with ai_extract

In [0]:
display(spark.sql("""
SELECT
  story_id,
  extracted.ticker AS ticker,
  extracted.event_type AS event_type,
  extracted.country AS country,
  extracted.policy_signal AS policy_signal,
  headline
FROM (
  SELECT
    story_id,
    headline,
    ai_extract(
      news_text,
      array('ticker', 'event_type', 'country', 'policy_signal')
    ) AS extracted
  FROM raw_news_ai_text
)
LIMIT 10
"""))

### 6. Short signal summaries with ai_summarize

In [0]:
display(spark.sql("""
SELECT
  story_id,
  extracted.ticker AS ticker,
  extracted.event_type AS event_type,
  extracted.country AS country,
  extracted.policy_signal AS policy_signal,
  headline
FROM (
  SELECT
    story_id,
    headline,
    ai_extract(
      news_text,
      array('ticker', 'event_type', 'country', 'policy_signal')
    ) AS extracted
  FROM raw_news_ai_text
)
LIMIT 50
"""))

In [0]:
display(spark.sql("""
SELECT
  story_id,
  extracted:response:ticker::STRING AS ticker,
  extracted:response:event_type::STRING AS event_type,
  extracted:response:country::STRING AS country,
  extracted:response:policy_signal::STRING AS policy_signal,
  headline
FROM (
  SELECT
    story_id,
    headline,
    ai_extract(
      news_text,
      -- Use a JSON-object schema with types + descriptions
      '{
        "ticker":        {"type": "string",  "description": "Primary stock or crypto ticker mentioned (e.g. AAPL, BTCUSD)"},
        "event_type":    {"type": "enum",    "labels": ["earnings", "macro", "regulation", "geopolitical", "other"], "description": "Category of the market event"},
        "country":       {"type": "string",  "description": "ISO-3166 alpha-2 country code most relevant to the news"},
        "policy_signal": {"type": "enum",    "labels": ["hawkish", "dovish", "neutral", "unknown"], "description": "Monetary/fiscal policy stance implied"}
      }',
      -- Global instructions to further steer the model
      MAP('instructions', 'You are a financial-news analyst. Extract only the single most relevant entity per field. If a field is not mentioned, return null.')
    ) AS extracted
  FROM raw_news_ai_text
)
LIMIT 50
"""))

### 7. Redaction with ai_mask

In [0]:
display(spark.sql("""
SELECT
  story_id,
  ai_mask(news_text, array('person', 'email', 'phone', 'address')) AS masked_text
FROM raw_news_ai_text
LIMIT 20
"""))

### 8. Translation with ai_translate

In [0]:
display(spark.sql("""
SELECT
  story_id,
  headline AS headline_en,
  ai_translate(headline, 'es') AS headline_es,
  ai_translate(headline, 'fr') AS headline_fr
FROM raw_news_ai_text
LIMIT 20
"""))

### 9. Near-duplicate headline detection with ai_similarity

In [0]:
display(spark.sql("""
WITH base AS (
  SELECT story_id, headline
  FROM raw_news_ai_text
  WHERE headline IS NOT NULL
  LIMIT 80
)
SELECT
  a.story_id AS story_id_a,
  b.story_id AS story_id_b,
  a.headline AS headline_a,
  b.headline AS headline_b,
  ai_similarity(a.headline, b.headline) AS similarity_score
FROM base a
JOIN base b
  ON a.story_id < b.story_id
WHERE ai_similarity(a.headline, b.headline) >= 0.4
ORDER BY similarity_score DESC
LIMIT 30
"""))

### 10. Free-form analyst note generation with ai_gen

In [0]:
display(spark.sql("""
WITH base AS (
  SELECT story_id, headline
  FROM raw_news_ai_text
  WHERE headline IS NOT NULL
  LIMIT 80
)
SELECT
  a.story_id AS story_id_a,
  b.story_id AS story_id_b,
  a.headline AS headline_a,
  b.headline AS headline_b,
  ai_similarity(a.headline, b.headline) AS similarity_score
FROM base a
JOIN base b
  ON a.story_id < b.story_id
WHERE ai_similarity(a.headline, b.headline) >= 0.88
ORDER BY similarity_score DESC
LIMIT 30
"""))

### 11. Advanced structured extraction with ai_query

In [0]:
if "MODEL_ENDPOINT" not in globals() or MODEL_ENDPOINT is None:
    raise RuntimeError("MODEL_ENDPOINT is not set. Run cell 9 first.")

print("Running ai_query with MODEL_ENDPOINT:", MODEL_ENDPOINT)

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW raw_news_ai_query AS
SELECT
  story_id,
  headline,
  ai_query(
    '{MODEL_ENDPOINT}',
    concat(
      'Extract one JSON object with keys: ',
      'ticker, direction, horizon_minutes, confidence_0_to_1, rationale_1_sentence. ',
      'direction must be one of bullish, bearish, neutral. ',
      'Text: ', news_text
    ),
    responseFormat => '{{"type":"json_object"}}'
  ) AS signal_json
FROM raw_news_ai_text
LIMIT 50
""")

display(spark.sql("""
SELECT story_id, headline, signal_json
FROM raw_news_ai_query
LIMIT 10
"""))

### 12. Parse ai_query JSON into typed columns

In [0]:
display(spark.sql("""
SELECT
  story_id,
  parsed.ticker AS ticker,
  parsed.direction AS direction,
  parsed.horizon_minutes AS horizon_minutes,
  parsed.confidence_0_to_1 AS confidence_0_to_1,
  parsed.rationale_1_sentence AS rationale
FROM (
  SELECT
    story_id,
    from_json(
      signal_json,
      'ticker STRING, direction STRING, horizon_minutes INT, confidence_0_to_1 DOUBLE, rationale_1_sentence STRING'
    ) AS parsed
  FROM raw_news_ai_query
)
LIMIT 50
"""))

### 13. One-pass feature table for downstream ML

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW raw_news_feature_table AS
SELECT
  story_id,
  published_at_ts,
  ai_analyze_sentiment(news_text) AS sentiment,
  ai_classify(news_text, array('bullish', 'bearish', 'neutral')) AS market_bias,
  ai_summarize(news_text, 40) AS summary_40w,
  ai_extract(news_text, array('ticker', 'event_type', 'country', 'policy_signal')) AS entities
FROM raw_news_ai_text
""")

display(spark.sql("""
SELECT
  story_id,
  published_at_ts,
  sentiment,
  market_bias,
  summary_40w,
  entities.ticker AS ticker,
  entities.event_type AS event_type,
  entities.country AS country,
  entities.policy_signal AS policy_signal
FROM raw_news_feature_table
LIMIT 30
"""))

### 15. Document parsing with ai_parse_document (companion pattern)

In [0]:
DOCS_VOLUME_PATH = "/Volumes/smart-trading-dev/00_volume/test-docs"

display(spark.sql(f"""
SELECT
  path,
  CAST(ai_parse_document(content) AS STRING) AS parsed_document
FROM read_files('{DOCS_VOLUME_PATH}', format => 'binaryFile')
LIMIT 1
"""))